# Linear Regression from First Principles
Accompanies **Chapter 7: Linear Regression from First Principles**.

Ordinary Least Squares (OLS) minimizes the Residual Sum of Squares:
$$\mathcal{L}(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 = (\mathbf{y} - \mathbf{X}\mathbf{w})^T (\mathbf{y} - \mathbf{X}\mathbf{w})$$
Setting the gradient with respect to $\mathbf{w}$ to zero yields the **Normal Equations**:
$$\mathbf{X}^T \mathbf{X} \mathbf{w}^* = \mathbf{X}^T \mathbf{y} \implies \mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In this notebook, we:
1. Implement the Normal Equations using numerically stable linear system solvers (`np.linalg.solve`).
2. Verify mathematical identity against `scikit-learn`'s `LinearRegression`.
3. Diagnose Gauss-Markov homoscedasticity assumptions using Residuals vs. Fitted value plots.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Ingest and clean lending dataset
data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)

df['Interest.Rate'] = pd.to_numeric(df['Interest.Rate'].astype(str).str.replace('%', ''), errors='coerce')
df['Debt.To.Income.Ratio'] = pd.to_numeric(df['Debt.To.Income.Ratio'].astype(str).str.replace('%', ''), errors='coerce')
df['FICO.Range'] = df['FICO.Range'].astype(str).str.split('-').apply(
    lambda x: (float(x[0]) + float(x[1])) / 2 if isinstance(x, list) and len(x) == 2 else np.nan
)

features = ['Amount.Requested', 'FICO.Range', 'Debt.To.Income.Ratio']
target = 'Interest.Rate'

clean_df = df[features + [target]].dropna()
X = clean_df[features].values
y = clean_df[target].values

print(f"Design matrix X: {X.shape}, Target vector y: {y.shape}")

### Augmenting the Design Matrix for the Intercept Term
The model is formulated as $\hat{y}_i = w_0 + \sum_{j=1}^p w_j x_{ij}$.
To represent the scalar intercept $w_0$ cleanly inside the matrix product $\mathbf{X}\mathbf{w}$, we augment $\mathbf{X}$ with a leading column of ones:
$$\mathbf{X}_{aug} = [\mathbf{1}, \mathbf{X}] \in \mathbb{R}^{n \times (p + 1)}$$

In [ ]:
n_samples = X.shape[0]
X_aug = np.column_stack([np.ones(n_samples), X])
X_aug.shape

### Numerical Stability: `np.linalg.solve` vs. Explicit Inversion
Computing $\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$ by directly inverting $\mathbf{X}^T \mathbf{X}$ with `np.linalg.inv` is numerically hazardous: it doubles the condition number (squaring floating-point precision loss) and costs $O(p^3)$ flops.
The standard numerical method treats $(\mathbf{X}^T \mathbf{X}) \mathbf{w}^* = \mathbf{X}^T \mathbf{y}$ as a linear system solved via Cholesky decomposition using `np.linalg.solve`.

In [ ]:
# Form Gram matrix and moment vector
A_gram = X_aug.T @ X_aug
b_moment = X_aug.T @ y

# Solve directly without matrix inversion
w_star = np.linalg.solve(A_gram, b_moment)

intercept_custom = w_star[0]
weights_custom = w_star[1:]

print(f"Custom Intercept: {intercept_custom:.6f}")
for name, w in zip(features, weights_custom):
    print(f"Weight ({name}): {w:.6f}")

### Verification Against Scikit-Learn

In [ ]:
sk_model = LinearRegression(fit_intercept=True)
sk_model.fit(X, y)

print(f"Sklearn Intercept: {sk_model.intercept_:.6f}")
for name, w in zip(features, sk_model.coef_):
    print(f"Weight ({name}): {w:.6f}")

# Verify exact numerical equivalence
assert np.isclose(intercept_custom, sk_model.intercept_, atol=1e-5)
assert np.allclose(weights_custom, sk_model.coef_, atol=1e-5)
print("\nExact numerical equivalence confirmed.")

### Diagnosing the Gauss-Markov Homoscedasticity Assumption
The Gauss-Markov Theorem proves OLS is the Best Linear Unbiased Estimator (BLUE) only when error variance is constant across all fitted values:
$$\text{Var}(\epsilon_i \mid \mathbf{x}_i) = \sigma^2$$
We plot residuals $e_i = y_i - \hat{y}_i$ against fitted predictions $\hat{y}_i$. A widening funnel shape signals heteroscedasticity, meaning standard error estimates and confidence intervals are biased.

In [ ]:
y_pred = sk_model.predict(X)
residuals = y - y_pred

plt.figure(figsize=(9, 4.8))
plt.scatter(y_pred, residuals, alpha=0.25, color='steelblue', edgecolors='none', s=18)
plt.axhline(0, color='crimson', linestyle='--', linewidth=1.5)
plt.xlabel('Fitted Values ($\\hat{y}$)', fontsize=11)
plt.ylabel('Residuals ($e = y - \\hat{y}$)', fontsize=11)
plt.title('Residuals vs. Fitted Values Diagnostic Plot', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Metric summary: R^2 and Root Mean Squared Error (RMSE)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"Coefficient of Determination (R^2): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE):        {rmse:.4f}")